# 章节练习

这一章覆盖了 SDPA 的接口与 dispatch、FlashAttention 的 IO 融合、GQA 的 KV head 缩减、稀疏注意力的结构化跳过、SFT document mask 的语义、TorchTitan dataloader 的文档边界、Configurable 机制、torch_npu 的 dispatcher，以及固定 `S=4096` 的性能计时和 trace。

以下题目均为判断题或选择题。作答时仍要区分配置、算子入口、设备事件和同步 wall-time 等不同层级的证据。

## 综合练习

1. （单选题）本章固定 B=2、S=4096、Nq=16、D=128、BF16。若 eager 路径物化 QK^T，逻辑 score 矩阵占用约多少 MiB？改用 GQA 且 Nkv=8 后，大小如何变化？
   A. 1,024 MiB；大小不变，因为 Q head 仍为 16
   B. 512 MiB；GQA 让 score 矩阵减半
   C. 2,048 MiB；GQA 让 score 矩阵翻倍
   D. 128 MiB；大小只由 Nkv 决定

2. （单选题）TorchTitan 的 attention 配置要切换到 FlexAttention，最应该修改哪一层？
   A. model forward 中临时替换张量
   B. attention registry 或 backend config 中的模块选择
   C. tokenizer 的词表
   D. optimizer 的学习率

3. （单选题）固定 S=4096 的 trace 中，SDPA 与 FusionAttention V3 的 device time 接近，但 CPU 调用链不同。该结果最多支持什么结论？
   A. 两条路径很可能落到相同或等价的融合设备路径；端到端训练速度仍需同步 wall-time 验证
   B. FusionAttention V3 一定比 SDPA 快约 1.2 倍
   C. 两条路径的显存占用一定完全相同
   D. 该模型已经在所有变长输入上完成验证

4. （判断题）benchmark 去掉 warmup、去掉计时前后的 `torch.npu.synchronize()`，只运行一次，就足以证明 FusionAttention V3 比 SDPA 快约 1.2 倍。

5. （单选题）profiler trace 中出现 `aten::scaled_dot_product_attention`，下面哪项判断正确？
   A. 它只是上层算子名，不能单独证明设备侧没有使用融合 kernel；还应检查设备事件
   B. 它出现就说明一定没有使用任何 NPU 融合算子
   C. kernel 数量越少就一定代表端到端训练越快
   D. 只看 CPU 调用链就能得到准确的 device time

6. （多选题）一份合格的固定 shape 性能结论至少应包含哪些信息？
   A. shape、dtype、硬件和软件版本
   B. 比较对象、forward 或 fwd+bwd 口径、warmup 和重复次数
   C. 同步边界、主要发现及适用范围限制
   D. 只写一个加速比，不说明测量方法

7. （判断题）固定长度 causal attention 得出的性能结论，可以不经重新验证直接外推到第 6 章的 packed VarLen 场景。

## 参考答案

先独立完成练习，再对照 [解释型参考答案](answer/05.06_answer.txt)。答案给出推理和证据边界，不只列结论。

In [ ]:
!cat ./answer/05.06_answer.txt
